## 1. Data Understanding

* Memuat dataset menggunakan pandas
* Menampilkan 5 baris pertama
* Menampilkan jumlah baris dan kolom
* Menampilkan nama kolom
* Mengidentifikasi variabel:

  * Target: Status Pesanan
  * Fitur numerik: data berbentuk angka
  * Fitur kategorikal: data berbentuk teks/kategori


In [7]:
import pandas as pd

# upload data
from google.colab import files
uploaded = files.upload()

# load data
df = pd.read_csv('all_months_clean.csv', sep=';')

# 5 data pertama
print("5 data pertama")
display(df.head())

# ukuran data
shape_df = pd.DataFrame({
    'keterangan': ['jumlah baris', 'jumlah kolom'],
    'nilai': [df.shape[0], df.shape[1]]
})

print("\nukuran data")
display(shape_df)

# nama kolom
kolom_df = pd.DataFrame(df.columns, columns=['nama kolom'])

print("\nnama kolom")
display(kolom_df)

# tipe data
tipe_df = pd.DataFrame({
    'kolom': df.columns,
    'tipe data': df.dtypes.values
})

print("\ntipe data")
display(tipe_df)

# klasifikasi fitur
numerik = df.select_dtypes(include=['int64','float64']).columns
kategorikal = df.select_dtypes(include=['object']).columns

fitur_df = pd.DataFrame({
    'jenis': ['numerik', 'kategorikal', 'target'],
    'kolom': [list(numerik), list(kategorikal), 'Status Pesanan']
})

print("\nklasifikasi fitur")
display(fitur_df)

Saving all_months_clean.csv to all_months_clean.csv
5 data pertama


,order_id,total_qty,total_weight_gr,total_returned_qty,Total Diskon,product_categories,num_product_categories,Status Pesanan,Alasan Pembatalan,Opsi Pengiriman,Metode Pembayaran,Kota/Kabupaten,Provinsi,Ongkos Kirim Dibayar oleh Pembeli,Estimasi Potongan Biaya Pengiriman,Total Pembayaran,Perkiraan Ongkos Kirim,Waktu Pesanan Dibuat,source_file
0,order_id,2,2000,0,0,Celengan,1,Selesai,NaN,Reguler (Cashless)-SPX Standard,Saldo ShopeePay,KOTA SERANG,BANTEN,0,10000,38300,10000,01/04/2024 00:15,AprilSales2024.xlsx
1,order_id,1,500,0,0,Celengan,1,Selesai,NaN,Hemat Kargo-SPX Hemat,COD (Bayar di Tempat),KOTA SEMARANG,JAWA TENGAH,0,14500,18576,14500,01/04/2024 01:47,AprilSales2024.xlsx
2,order_id,1,500,0,0,Celengan,1,Selesai,NaN,Hemat Kargo-SPX Hemat,SeaBank Bayar Instan,KAB. BOGOR,JAWA BARAT,0,8000,7069,8000,01/04/2024 04:25,AprilSales2024.xlsx
3,order_id,2,400,0,0,Mangkok Sambal / Saus,1,Selesai,NaN,Hemat Kargo-SPX Hemat,COD (Bayar di Tempat),KOTA JAMBI,JAMBI,0,20000,32200,20000,01/04/2024 04:41,AprilSales2024.xlsx
4,order_id,3,3600,0,0,"Keranjang, Other, Tempat Nasi",3,Batal,Dibatalkan oleh Pembeli. Alasan: Ubah Pesanan ...,Hemat Kargo-SPX Hemat,COD (Bayar di Tempat),KOTA TANGERANG,BANTEN,0,0,0,8000,01/04/2024 06:12,AprilSales2024.xlsx



ukuran data


,keterangan,nilai
0,jumlah baris,20848
1,jumlah kolom,19



nama kolom


,nama kolom
0,order_id
1,total_qty
2,total_weight_gr
3,total_returned_qty
4,Total Diskon
5,product_categories
6,num_product_categories
7,Status Pesanan
8,Alasan Pembatalan
9,Opsi Pengiriman



tipe data


,kolom,tipe data
0,order_id,object
1,total_qty,int64
2,total_weight_gr,int64
3,total_returned_qty,int64
4,Total Diskon,int64
5,product_categories,object
6,num_product_categories,int64
7,Status Pesanan,object
8,Alasan Pembatalan,object
9,Opsi Pengiriman,object



klasifikasi fitur


,jenis,kolom
0,numerik,"[total_qty, total_weight_gr, total_returned_qt..."
1,kategorikal,"[order_id, product_categories, Status Pesanan,..."
2,target,Status Pesanan


## 2. Data Preprocessing

* Membuat fitur baru:

  * is_cancelled dari Status Pesanan
  * is_cod dari Metode Pembayaran
* Mengekstrak informasi waktu:

  * Mengambil jam (hour) dari Waktu Pesanan Dibuat
* Membuat fitur tambahan:

  * time_category berdasarkan jam (pagi, siang, malam, dini_hari)
* Menyimpan dataset hasil preprocessing dengan nama:

  * data_processed_24782059.csv


In [8]:
# Copy data
df_processed = df.copy()

# Fitur pembatalan
df_processed['is_cancelled'] = df_processed['Status Pesanan'].apply(
    lambda x: 1 if 'batal' in str(x).lower() else 0
)

# Fitur COD
df_processed['is_cod'] = df_processed['Metode Pembayaran'].apply(
    lambda x: 1 if 'cod' in str(x).lower() else 0
)

# Ambil jam
df_processed['order_time'] = pd.to_datetime(df_processed['Waktu Pesanan Dibuat'], errors='coerce')
df_processed['hour'] = df_processed['order_time'].dt.hour

# Kategori waktu
def kategori_waktu(hour):
    if pd.isna(hour):
        return 'unknown'
    elif 5 <= hour < 12:
        return 'pagi'
    elif 12 <= hour < 18:
        return 'siang'
    elif 18 <= hour < 24:
        return 'malam'
    else:
        return 'dini_hari'

df_processed['time_category'] = df_processed['hour'].apply(kategori_waktu)

# Simpan data
df_processed.to_csv('data_processed_24782059.csv', index=False)

# Output ringkas
display(df_processed[['Status Pesanan','is_cancelled',
                      'Metode Pembayaran','is_cod',
                      'hour','time_category']].head(10))

print("\nPembatalan:")
print(df_processed['is_cancelled'].value_counts())

,Status Pesanan,is_cancelled,Metode Pembayaran,is_cod,hour,time_category
0,Selesai,0,Saldo ShopeePay,0,0.0,dini_hari
1,Selesai,0,COD (Bayar di Tempat),1,1.0,dini_hari
2,Selesai,0,SeaBank Bayar Instan,0,4.0,dini_hari
3,Selesai,0,COD (Bayar di Tempat),1,4.0,dini_hari
4,Batal,1,COD (Bayar di Tempat),1,6.0,pagi
5,Selesai,0,SPayLater,0,7.0,pagi
6,Batal,1,SPayLater,0,7.0,pagi
7,Selesai,0,Online Payment,0,7.0,pagi
8,Batal,1,Online Payment,0,7.0,pagi
9,Selesai,0,Online Payment,0,8.0,pagi



Pembatalan:
is_cancelled
0    18018
1     2830
Name: count, dtype: int64


## 3. Feature Exploration

* Memilih fitur yang dianalisis:

  * Metode Pembayaran (is_cod)
  * Perkiraan Ongkos Kirim
  * Jumlah Produk (total_qty)

* Hubungan logis:

  * is_cod → metode pembayaran dapat mempengaruhi keputusan pembatalan
  * Ongkir → ongkos kirim tinggi dapat meningkatkan kemungkinan pembatalan
  * Jumlah produk → jumlah barang dapat mempengaruhi tingkat komitmen pembeli

* Alasan pemilihan:

  * Fitur memiliki hubungan langsung dengan keputusan pembelian
  * Fitur tersedia dalam dataset dan dapat dianalisis secara kuantitatif
  * Fitur menunjukkan variasi yang dapat mempengaruhi pembatalan pesanan


In [9]:
# =======================
# 1. COD
# =======================
print("=== TABEL COD (Preview) ===")
display(df_processed[['Metode Pembayaran','is_cod','Status Pesanan','is_cancelled']].head(10))

print("\n=== ANALISIS COD ===")
cod_analysis = df_processed.groupby('is_cod')['is_cancelled'].mean()
print("Rata-rata:")
print(cod_analysis)

cod_table = pd.crosstab(df_processed['is_cod'], df_processed['is_cancelled'], normalize='index') * 100
print("\nPersentase:")
print(cod_table)


# =======================
# 2. ONGKIR
# =======================
# Kategori ongkir
df_processed['ongkir_kategori'] = pd.qcut(
    df_processed['Perkiraan Ongkos Kirim'],
    3,
    labels=['Murah', 'Sedang', 'Mahal'],
    duplicates='drop'
)

print("\n=== TABEL ONGKIR (Preview) ===")
display(df_processed[['Perkiraan Ongkos Kirim','ongkir_kategori','Status Pesanan','is_cancelled']].head(10))

print("\n=== ANALISIS ONGKIR ===")
ongkir_analysis = df_processed.groupby('is_cancelled')['Perkiraan Ongkos Kirim'].mean()
print("Rata-rata:")
print(ongkir_analysis)

ongkir_table = pd.crosstab(df_processed['ongkir_kategori'], df_processed['is_cancelled'], normalize='index') * 100
print("\nPersentase:")
print(ongkir_table)


# =======================
# 3. JUMLAH PRODUK
# =======================
# Kategori qty (manual)
def kategori_qty(qty):
    if qty <= 1:
        return 'Sedikit'
    elif qty <= 3:
        return 'Sedang'
    else:
        return 'Banyak'

df_processed['qty_kategori'] = df_processed['total_qty'].apply(kategori_qty)

print("\n=== TABEL JUMLAH PRODUK (Preview) ===")
display(df_processed[['total_qty','qty_kategori','Status Pesanan','is_cancelled']].head(10))

print("\n=== ANALISIS JUMLAH PRODUK ===")
qty_analysis = df_processed.groupby('is_cancelled')['total_qty'].mean()
print("Rata-rata:")
print(qty_analysis)

qty_table = pd.crosstab(df_processed['qty_kategori'], df_processed['is_cancelled'], normalize='index') * 100
print("\nPersentase:")
print(qty_table)

=== TABEL COD (Preview) ===


,Metode Pembayaran,is_cod,Status Pesanan,is_cancelled
0,Saldo ShopeePay,0,Selesai,0
1,COD (Bayar di Tempat),1,Selesai,0
2,SeaBank Bayar Instan,0,Selesai,0
3,COD (Bayar di Tempat),1,Selesai,0
4,COD (Bayar di Tempat),1,Batal,1
5,SPayLater,0,Selesai,0
6,SPayLater,0,Batal,1
7,Online Payment,0,Selesai,0
8,Online Payment,0,Batal,1
9,Online Payment,0,Selesai,0



=== ANALISIS COD ===
Rata-rata:
is_cod
0    0.138131
1    0.133819
Name: is_cancelled, dtype: float64

Persentase:
is_cancelled          0          1
is_cod                            
0             86.186896  13.813104
1             86.618131  13.381869

=== TABEL ONGKIR (Preview) ===


,Perkiraan Ongkos Kirim,ongkir_kategori,Status Pesanan,is_cancelled
0,10000,Sedang,Selesai,0
1,14500,Sedang,Selesai,0
2,8000,Murah,Selesai,0
3,20000,Mahal,Selesai,0
4,8000,Murah,Batal,1
5,10000,Sedang,Selesai,0
6,16000,Sedang,Batal,1
7,8000,Murah,Selesai,0
8,16500,Sedang,Batal,1
9,33000,Mahal,Selesai,0



=== ANALISIS ONGKIR ===
Rata-rata:
is_cancelled
0    17301.082029
1    25585.357597
Name: Perkiraan Ongkos Kirim, dtype: float64

Persentase:
is_cancelled             0          1
ongkir_kategori                      
Murah            89.761571  10.238429
Sedang           87.957272  12.042728
Mahal            81.449739  18.550261

=== TABEL JUMLAH PRODUK (Preview) ===


,total_qty,qty_kategori,Status Pesanan,is_cancelled
0,2,Sedang,Selesai,0
1,1,Sedikit,Selesai,0
2,1,Sedikit,Selesai,0
3,2,Sedang,Selesai,0
4,3,Sedang,Batal,1
5,2,Sedang,Selesai,0
6,1,Sedikit,Batal,1
7,1,Sedikit,Selesai,0
8,1,Sedikit,Batal,1
9,6,Banyak,Selesai,0



=== ANALISIS JUMLAH PRODUK ===
Rata-rata:
is_cancelled
0    2.485459
1    3.040636
Name: total_qty, dtype: float64

Persentase:
is_cancelled          0          1
qty_kategori                      
Banyak        84.315226  15.684774
Sedang        88.710223  11.289777
Sedikit       85.778030  14.221970


## 4. Analisis Pola

* COD vs non-COD:

  * Menampilkan tabel data dan persentase pembatalan
  * Hasil menunjukkan perbedaan kecil antara COD dan non-COD
  * Kesimpulan: metode pembayaran tidak berpengaruh signifikan

* Ongkir vs pembatalan:

  * Mengelompokkan ongkir menjadi murah, sedang, mahal
  * Menampilkan tabel dan persentase pembatalan
  * Kesimpulan: semakin mahal ongkir, semakin tinggi pembatalan

* Jumlah produk vs pembatalan:

  * Mengelompokkan jumlah produk menjadi sedikit, sedang, banyak
  * Menampilkan tabel dan persentase pembatalan
  * Kesimpulan: jumlah produk lebih banyak cenderung lebih sering dibatalkan


In [10]:
# a. COD vs pembatalan
print("tabel cod")
display(df_processed[['Metode Pembayaran','is_cod','Status Pesanan','is_cancelled']].head(10))

cod_table = pd.crosstab(df_processed['is_cod'], df_processed['is_cancelled'], normalize='index') * 100

print("\npersentase cod")
display(cod_table)

print("interpretasi: cod tidak berpengaruh signifikan")


# b. ongkir vs pembatalan
df_processed['ongkir_kategori'] = pd.qcut(
    df_processed['Perkiraan Ongkos Kirim'],
    3,
    labels=['murah','sedang','mahal'],
    duplicates='drop'
)

print("\ntabel ongkir")
display(df_processed[['Perkiraan Ongkos Kirim','ongkir_kategori','Status Pesanan','is_cancelled']].head(10))

ongkir_table = pd.crosstab(df_processed['ongkir_kategori'], df_processed['is_cancelled'], normalize='index') * 100

print("\npersentase ongkir")
display(ongkir_table)

print("interpretasi: ongkir makin mahal → pembatalan meningkat")


# c. jumlah produk vs pembatalan
def kategori_qty(qty):
    if qty <= 1:
        return 'sedikit'
    elif qty <= 3:
        return 'sedang'
    else:
        return 'banyak'

df_processed['qty_kategori'] = df_processed['total_qty'].apply(kategori_qty)

print("\ntabel jumlah produk")
display(df_processed[['total_qty','qty_kategori','Status Pesanan','is_cancelled']].head(10))

qty_table = pd.crosstab(df_processed['qty_kategori'], df_processed['is_cancelled'], normalize='index') * 100

print("\npersentase jumlah produk")
display(qty_table)

print("interpretasi: jumlah produk lebih banyak → cenderung lebih sering batal")

tabel cod


,Metode Pembayaran,is_cod,Status Pesanan,is_cancelled
0,Saldo ShopeePay,0,Selesai,0
1,COD (Bayar di Tempat),1,Selesai,0
2,SeaBank Bayar Instan,0,Selesai,0
3,COD (Bayar di Tempat),1,Selesai,0
4,COD (Bayar di Tempat),1,Batal,1
5,SPayLater,0,Selesai,0
6,SPayLater,0,Batal,1
7,Online Payment,0,Selesai,0
8,Online Payment,0,Batal,1
9,Online Payment,0,Selesai,0



persentase cod


is_cancelled,0,1
is_cod,,
0,86.186896,13.813104
1,86.618131,13.381869


interpretasi: cod tidak berpengaruh signifikan

tabel ongkir


...

interpretasi: ongkir makin mahal → pembatalan meningkat

tabel jumlah produk


,total_qty,qty_kategori,Status Pesanan,is_cancelled
0,2,sedang,Selesai,0
1,1,sedikit,Selesai,0
2,1,sedikit,Selesai,0
3,2,sedang,Selesai,0
4,3,sedang,Batal,1
5,2,sedang,Selesai,0
6,1,sedikit,Batal,1
7,1,sedikit,Selesai,0
8,1,sedikit,Batal,1
9,6,banyak,Selesai,0



persentase jumlah produk


is_cancelled,0,1
qty_kategori,,
banyak,84.315226,15.684774
sedang,88.710223,11.289777
sedikit,85.778030,14.221970


interpretasi: jumlah produk lebih banyak → cenderung lebih sering batal


## 5. Segmentasi Data

* Membagi data berdasarkan ongkos kirim:

  * murah, sedang, mahal

* Menghitung tingkat pembatalan pada setiap kelompok

  * Menggunakan rata-rata is_cancelled dalam persen

* Menentukan kelompok dengan risiko tertinggi

  * Kelompok dengan persentase pembatalan paling besar (ongkir mahal)


In [11]:
import pandas as pd

# segmentasi berdasarkan ongkir
df_processed['ongkir_kategori'] = pd.qcut(
    df_processed['Perkiraan Ongkos Kirim'],
    3,
    labels=['murah','sedang','mahal'],
    duplicates='drop'
)

print("tabel segmentasi ongkir")
display(df_processed[['Perkiraan Ongkos Kirim','ongkir_kategori','is_cancelled']].head(10))

# hitung tingkat pembatalan
segmentasi = df_processed.groupby('ongkir_kategori')['is_cancelled'].mean() * 100

print("\ntingkat pembatalan (%)")
display(segmentasi)

# cari yang paling tinggi
tertinggi = segmentasi.idxmax()
nilai_tertinggi = segmentasi.max()

print("\nrisiko tertinggi:")
print(tertinggi, "=", nilai_tertinggi, "%")

tabel segmentasi ongkir


,Perkiraan Ongkos Kirim,ongkir_kategori,is_cancelled
0,10000,sedang,0
1,14500,sedang,0
2,8000,murah,0
3,20000,mahal,0
4,8000,murah,1
5,10000,sedang,0
6,16000,sedang,1
7,8000,murah,0
8,16500,sedang,1
9,33000,mahal,0



tingkat pembatalan (%)


/tmp/ipykernel_485/3900236937.py:15: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  segmentasi = df_processed.groupby('ongkir_kategori')['is_cancelled'].mean() * 100


,is_cancelled
ongkir_kategori,
murah,10.238429
sedang,12.042728
mahal,18.550261



risiko tertinggi:
mahal = 18.550261475886114 %


## 6. Analisis Konflik

* Data sesuai pola:

  * Ongkir mahal dan pesanan dibatalkan
  * Sesuai dengan pola bahwa ongkir tinggi meningkatkan pembatalan

* Data bertentangan:

  * Ongkir mahal tetapi pesanan tidak dibatalkan
  * Menunjukkan adanya faktor lain yang mempengaruhi keputusan pelanggan


In [12]:
# 1. data yang sesuai pola (ongkir mahal & batal)
sesuai = df_processed[
    (df_processed['ongkir_kategori'] == 'mahal') &
    (df_processed['is_cancelled'] == 1)
]

print("data sesuai pola")
display(sesuai[['Perkiraan Ongkos Kirim','ongkir_kategori','Status Pesanan','is_cancelled']].head(1))


# 2. data yang bertentangan (ongkir mahal tapi tidak batal)
konflik = df_processed[
    (df_processed['ongkir_kategori'] == 'mahal') &
    (df_processed['is_cancelled'] == 0)
]

print("\ndata bertentangan")
display(konflik[['Perkiraan Ongkos Kirim','ongkir_kategori','Status Pesanan','is_cancelled']].head(1))

data sesuai pola


,Perkiraan Ongkos Kirim,ongkir_kategori,Status Pesanan,is_cancelled
10,53000,mahal,Batal,1



data bertentangan


,Perkiraan Ongkos Kirim,ongkir_kategori,Status Pesanan,is_cancelled
3,20000,mahal,Selesai,0
